# Punto 4 — Análisis individual: Lionel Messi

**Proyecto Final · Curso LanusStats** · Lucas Marinelli · @datafutbol_ar

Partido: **Argentina 1–2 Arabia Saudita** — debut Mundial 2022 (`match_id 3857300`).

> **Consigna oficial.** Elegir un jugador y hacer:
> 1. **Mapa de calor** (heatmap) — dónde estuvo
> 2. **Mapa de tiros** — qué intentó
> 3. **Mapa de acciones** — recuperaciones, pases, faltas, intercepciones

**Jugador elegido**: Lionel Messi (capitán, juega en ambos partidos del proyecto).


## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from helpers import *
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from mplsoccer import Pitch, VerticalPitch

# === Paleta semáforo CVD-safe (Wong 2011) ===
COLOR_OK     = '#009E73'   # verde-teal
COLOR_NO_OK  = '#B36930'   # naranja-rojo (ajuste de Lucas)

# === Colormap "datafutbol" para heatmap ===
cmap_marca = LinearSegmentedColormap.from_list(
    'datafutbol', [COLORS['bg'], COLORS['primary'], COLORS['accent']]
)

ev = cargar_eventos(MATCH_ARG_SAU, 'arg_sau')
ev = añadir_xy(ev)
print('eventos:', ev.shape)


## 1. Filtrar eventos de Messi

In [ ]:
JUGADOR = 'Lionel Andrés Messi Cuccittini'

ev_messi = ev[ev['player'] == JUGADOR].copy()
print(f'Eventos de Messi: {len(ev_messi)}')
print()
print(ev_messi['type'].value_counts())


## 2. Función: mapa de calor (heatmap)

In [ ]:
def heatmap_jugador(ev_jug, jugador, nombre_archivo):
    """Mapa de calor de un jugador (densidad de eventos)."""
    pos = ev_jug.dropna(subset=['x', 'y'])
    print(f'{jugador}: {len(pos)} eventos con posición')

    pitch = Pitch(pitch_type='statsbomb',
                  pitch_color=COLORS['bg'],
                  line_color=COLORS['text'],
                  linewidth=1.5,
                  line_zorder=2,
                  pad_top=15)
    fig, ax = pitch.draw(figsize=(11, 7.5))
    fig.patch.set_facecolor(COLORS['bg'])

    bin_stat = pitch.bin_statistic(pos.x, pos.y,
                                    statistic='count',
                                    bins=(12, 8))
    pcm = pitch.heatmap(bin_stat, ax=ax, cmap=cmap_marca,
                        edgecolors=COLORS['bg'], alpha=0.85, zorder=1)

    nombre_corto = jugador.split()[-1] if 'Messi' not in jugador else 'Messi'
    ax.set_title(f'Mapa de calor · {nombre_corto}',
                 color=COLORS['text'], fontsize=16, weight='bold',
                 loc='left', pad=18)
    ax.text(0, 86,
            f'{len(pos)} eventos · Argentina vs Arabia Saudita',
            color=COLORS['accent'], fontsize=11, style='italic')

    cbar = fig.colorbar(pcm, ax=ax, shrink=0.7, pad=0.02)
    cbar.set_label('Cantidad de eventos', color=COLORS['text'], fontsize=10)
    cbar.ax.tick_params(colors=COLORS['text'], labelsize=9)
    cbar.outline.set_edgecolor(COLORS['accent'])

    watermark(fig, 'Datos: StatsBomb Open Data')
    guardar_fig(fig, nombre_archivo)
    plt.show()


## 3. Aplicar heatmap a Messi

In [ ]:
heatmap_jugador(ev_messi, JUGADOR, 'punto4_heatmap_messi')

## 4. Función: mapa de tiros del jugador

> 🔧 **Ajuste v2:** multiplicador del tamaño bajó de 1500 a **500**, porque el penal de Messi
> (xG ≈ 0.79) hacía que el círculo dorado tapara casi todo el área. Ahora la escala
> respeta la proporción entre tiros chicos y el penal.


In [ ]:
def tiros_jugador(ev_jug, jugador, nombre_archivo):
    """Shot map de un jugador específico."""
    tiros = ev_jug[ev_jug['type'] == 'Shot'].copy()
    if tiros.empty:
        print(f'{jugador} no tuvo tiros en este partido')
        return None

    es_gol = tiros['shot_outcome'] == 'Goal'
    print(f'{jugador}: {len(tiros)} tiros · {es_gol.sum()} goles')

    pitch = VerticalPitch(pitch_type='statsbomb',
                          pitch_color=COLORS['bg'],
                          line_color=COLORS['text'],
                          linewidth=1.5,
                          half=True,
                          pad_top=2)
    fig, ax = pitch.draw(figsize=(6, 8.5))
    fig.patch.set_facecolor(COLORS['bg'])

    def size_xg(xg):
        # ↓ AJUSTAR ACÁ si querés círculos más grandes/chicos:
        #   - subir multiplicador → todo más grande
        #   - subir base (+80) → los tiros chicos se ven más
        return xg * 500 + 80

    # Tiros NO gol
    pitch.scatter(
        tiros.loc[~es_gol, 'x'], tiros.loc[~es_gol, 'y'],
        s=size_xg(tiros.loc[~es_gol, 'shot_statsbomb_xg']),
        ax=ax, color=COLORS['primary'], edgecolors=COLORS['text'],
        linewidth=1.5, alpha=0.85, zorder=2
    )
    # Tiros gol
    if es_gol.any():
        pitch.scatter(
            tiros.loc[es_gol, 'x'], tiros.loc[es_gol, 'y'],
            s=size_xg(tiros.loc[es_gol, 'shot_statsbomb_xg']),
            ax=ax, color=COLORS['accent'], edgecolors=COLORS['text'],
            linewidth=2.5, alpha=0.95, zorder=3
        )

    xg_total = tiros['shot_statsbomb_xg'].sum()
    fig.suptitle(f'Mapa de tiros · {jugador.split()[-1] if "Messi" not in jugador else "Messi"}',
                 color=COLORS['text'], fontsize=16, weight='bold', y=0.96)
    fig.text(0.5, 0.92,
             f'{len(tiros)} tiros · {es_gol.sum()} goles · {xg_total:.2f} xG',
             ha='center', color=COLORS['accent'], fontsize=11, style='italic')

    # Doble leyenda
    color_handles = [
        Line2D([0], [0], marker='o', linestyle='',
               markerfacecolor=COLORS['primary'], markeredgecolor=COLORS['text'],
               markeredgewidth=1.2, markersize=14, label='Sin gol'),
        Line2D([0], [0], marker='o', linestyle='',
               markerfacecolor=COLORS['accent'], markeredgecolor=COLORS['text'],
               markeredgewidth=1.5, markersize=14, label='Gol'),
    ]
    leg1 = ax.legend(handles=color_handles,
                     loc='upper left', bbox_to_anchor=(1.02, 0.98),
                     facecolor=COLORS['bg'], edgecolor=COLORS['accent'],
                     labelcolor=COLORS['text'], fontsize=11, framealpha=0.95,
                     title='Resultado', title_fontsize=11, labelspacing=1.0)
    plt.setp(leg1.get_title(), color=COLORS['text'], weight='bold')
    ax.add_artist(leg1)

    xg_values = [0.05, 0.20, 0.50]
    size_handles = [
        Line2D([0], [0], marker='o', linestyle='',
               markerfacecolor=COLORS['primary'], markeredgecolor=COLORS['text'],
               markeredgewidth=1, markersize=(size_xg(v) ** 0.5) * 0.7,
               label=f'{v:.2f} xG')
        for v in xg_values
    ]
    leg2 = ax.legend(handles=size_handles,
                     loc='upper left', bbox_to_anchor=(1.02, 0.45),
                     facecolor=COLORS['bg'], edgecolor=COLORS['accent'],
                     labelcolor=COLORS['text'], fontsize=10, framealpha=0.95,
                     title='Tamaño = xG', title_fontsize=11,
                     labelspacing=1.6, handletextpad=1.5, borderpad=1.2)
    plt.setp(leg2.get_title(), color=COLORS['text'], weight='bold')

    watermark(fig, 'Datos: StatsBomb Open Data')
    guardar_fig(fig, nombre_archivo)
    plt.show()
    return tiros


## 5. Aplicar tiros a Messi

In [ ]:
tiros_messi = tiros_jugador(ev_messi, JUGADOR, 'punto4_tiros_messi')

## 6. Función: mapa de acciones

Combina:
- 🔵 **Recuperaciones** (Ball Recovery)
- 🟢 **Intercepciones** (Interception)
- 🟠 **Faltas cometidas** (Foul Committed)
- 🟡 **Pases progresivos** (Pass con avance ≥10m)


In [ ]:
def acciones_jugador(ev_jug, jugador, nombre_archivo):
    """Mapa multi-evento de un jugador."""
    pitch = Pitch(pitch_type='statsbomb',
                  pitch_color=COLORS['bg'],
                  line_color=COLORS['text'],
                  linewidth=1.5,
                  pad_top=18)
    fig, ax = pitch.draw(figsize=(11, 7.5))
    fig.patch.set_facecolor(COLORS['bg'])

    def _xy(loc, i):
        if isinstance(loc, (list, tuple, np.ndarray)) and len(loc) > i:
            return loc[i]
        return np.nan

    # Pases progresivos (flechas dorado de fondo)
    pases = ev_jug[ev_jug['type'] == 'Pass'].copy()
    pases['x_end'] = pases['pass_end_location'].apply(lambda l: _xy(l, 0))
    pases['y_end'] = pases['pass_end_location'].apply(lambda l: _xy(l, 1))
    pases = pases.dropna(subset=['x_end', 'y_end'])
    prog = pases[(pases['x_end'] - pases['x']) > 10]
    completados = prog['pass_outcome'].isna()
    n_prog = len(prog)

    if completados.any():
        pitch.arrows(
            prog.loc[completados, 'x'], prog.loc[completados, 'y'],
            prog.loc[completados, 'x_end'], prog.loc[completados, 'y_end'],
            ax=ax, width=1.3, headwidth=4, color=COLORS['accent'],
            alpha=0.45, zorder=1
        )

    # Recuperaciones
    recup = ev_jug[ev_jug['type'] == 'Ball Recovery'].dropna(subset=['x', 'y'])
    if not recup.empty:
        pitch.scatter(recup.x, recup.y, ax=ax,
                       marker='o', s=200, color=COLORS['primary'],
                       edgecolors=COLORS['text'], linewidth=1.5,
                       alpha=0.95, zorder=4)

    # Intercepciones
    inter = ev_jug[ev_jug['type'] == 'Interception'].dropna(subset=['x', 'y'])
    if not inter.empty:
        pitch.scatter(inter.x, inter.y, ax=ax,
                       marker='^', s=250, color=COLOR_OK,
                       edgecolors=COLORS['text'], linewidth=1.5,
                       alpha=0.95, zorder=4)

    # Faltas
    faltas = ev_jug[ev_jug['type'] == 'Foul Committed'].dropna(subset=['x', 'y'])
    if not faltas.empty:
        pitch.scatter(faltas.x, faltas.y, ax=ax,
                       marker='X', s=250, color=COLOR_NO_OK,
                       edgecolors=COLORS['text'], linewidth=1.5,
                       alpha=0.95, zorder=4)

    nombre_corto = jugador.split()[-1] if 'Messi' not in jugador else 'Messi'
    ax.set_title(f'Mapa de acciones · {nombre_corto}',
                 color=COLORS['text'], fontsize=16, weight='bold',
                 loc='left', pad=20)
    ax.text(0, 86,
            f'{len(recup)} recuperaciones · {len(inter)} intercepciones · '
            f'{len(faltas)} faltas · {n_prog} pases progresivos',
            color=COLORS['accent'], fontsize=10.5, style='italic')

    legend_handles = [
        Line2D([0], [0], marker='o', linestyle='', markersize=12,
               markerfacecolor=COLORS['primary'], markeredgecolor=COLORS['text'],
               label='Recuperación'),
        Line2D([0], [0], marker='^', linestyle='', markersize=13,
               markerfacecolor=COLOR_OK, markeredgecolor=COLORS['text'],
               label='Intercepción'),
        Line2D([0], [0], marker='X', linestyle='', markersize=12,
               markerfacecolor=COLOR_NO_OK, markeredgecolor=COLORS['text'],
               label='Falta cometida'),
        Line2D([0], [0], marker=r'$\rightarrow$', linestyle='', markersize=14,
               markerfacecolor=COLORS['accent'], markeredgecolor=COLORS['accent'],
               label='Pase progresivo'),
    ]
    ax.legend(handles=legend_handles,
              loc='upper right', bbox_to_anchor=(1, 1.14),
              facecolor=COLORS['bg'], edgecolor=COLORS['accent'],
              labelcolor=COLORS['text'], fontsize=10, framealpha=0.95,
              ncol=4, handletextpad=0.4, columnspacing=0.8)

    watermark(fig, 'Datos: StatsBomb Open Data')
    guardar_fig(fig, nombre_archivo)
    plt.show()
    return {'recup': len(recup), 'inter': len(inter),
            'faltas': len(faltas), 'pases_prog': n_prog}


## 7. Aplicar acciones a Messi

In [ ]:
stats_messi = acciones_jugador(ev_messi, JUGADOR, 'punto4_acciones_messi')
print(stats_messi)

## Resumen — Punto 4 ✅

**Mapa de calor:** Messi operando en media-cancha y borde del área (zona natural de creación).

**Mapa de tiros:** 4 tiros · 1 gol · ~1.07 xG. El penal (círculo dorado grande) abre el marcador.

**Mapa de acciones:** combina pases progresivos (territorio) + recuperaciones + faltas + intercepciones.

> **Hilo conductor del proyecto:** este Messi de ARG-SAU es el **mismo** que va a hacer 2 goles
> + asistencia en la final ARG-FRA. Comparar los dos mapas (mismo jugador, distinto resultado
> del equipo) es **el post que más va a circular** del proyecto.

---

### Decisiones de diseño

| Decisión | Por qué |
|---|---|
| Heatmap con cmap `datafutbol` (navy → celeste → dorado) | Coherente con identidad visual |
| VerticalPitch para shot map | Estándar industria (igual que Punto 3) |
| **Multiplicador xG = 500** (era 1500) | El penal (xG~0.79) tapaba todo el área. 500 mantiene proporción |
| 4 marker shapes en mapa de acciones | Legible incluso en blanco y negro |
| Pases progresivos en fondo (alpha 0.45) | No tapan los markers, dan contexto territorial |
